# Практика 5. Шкали вимірювання і частотний розподіл

**Студент:** Yurii_huz  
**Група:** IT-41  
**Варіант:** 1, Київ

Мета: класифікувати змінні за шкалами вимірювання, побудувати варіаційний ряд, частотну таблицю та гістограми для місячної кількості опадів.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

months = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]
precipitation = pd.Series([47, 44, 38, 43, 60, 78, 88, 70, 48, 41, 47, 50], index=months, name="опади_мм")
print(precipitation)

## Завдання 1. Класифікація змінних за шкалою вимірювання

- `місто` — **номінальна шкала**, бо це назва категорії без природного порядку. Для неї коректна міра положення — мода.
- `номер місяця` — **порядкова шкала** у цьому завданні: місяці мають природний порядок, але нульовий місяць не означає відсутність місяця. Тому для нього коректною мірою є мода; середнє та медіана потребують обережної календарної інтерпретації. Водночас можна аргументувати інтервальну шкалу, якщо розглядати номери як рівновіддалені позиції календаря.
- `кількість опадів у мм` — **відносна шкала**, бо має природний нуль і змістовні відношення, наприклад 80 мм — удвічі більше за 40 мм. Для неї коректні середнє, медіана та мода.

In [ ]:
scales = pd.DataFrame({
    "змінна": ["місто", "номер місяця", "кількість опадів, мм"],
    "шкала": ["номінальна", "порядкова (можливе інтервальне трактування)", "відносна"],
    "коректні міри": ["мода", "переважно мода", "середнє, медіана, мода"],
})
print(scales)

## Завдання 2. Варіаційний ряд і частотна таблиця

In [ ]:
variation_series = np.sort(precipitation.to_numpy())
print("Варіаційний ряд:", variation_series)

bins = [35, 50, 65, 80, 90]
intervals = pd.cut(precipitation, bins=bins, include_lowest=True, right=True)
frequency = intervals.value_counts(sort=False)
frequency_table = pd.DataFrame({
    "абсолютна": frequency,
    "відносна_%": (frequency / len(precipitation) * 100).round(1),
    "накопичена": frequency.cumsum(),
})
print(frequency_table)

third_interval_cumulative = int(frequency_table["накопичена"].iloc[2])
third_interval_share = third_interval_cumulative / len(precipitation) * 100
print("Частка місяців не вище межі третього інтервалу (80 мм):", round(third_interval_share, 1), "%")

На межі третього інтервалу `80 мм` накопичена частота дорівнює 11 із 12 місяців, тобто приблизно `91,7%`. Це означає, що у 11 місяцях кількість опадів не перевищує 80 мм.

## Завдання 3. Гістограма

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(precipitation, edgecolor="black")
axes[0].set_title("За замовчуванням")
axes[0].set_xlabel("Опади, мм")
axes[0].set_ylabel("Частота")

axes[1].hist(precipitation, bins=3, edgecolor="black")
axes[1].set_title("3 інтервали")
axes[1].set_xlabel("Опади, мм")

axes[2].hist(precipitation, bins=10, edgecolor="black")
axes[2].set_title("10 інтервалів")
axes[2].set_xlabel("Опади, мм")

plt.tight_layout()
plt.show()

n = len(precipitation)
sqrt_bins = np.sqrt(n)
sturges_bins = 1 + np.log2(n)
print("Правило √n:", round(sqrt_bins, 2), "приблизно 3–4 інтервали")
print("Правило Стерджеса:", round(sturges_bins, 2), "приблизно 5 інтервалів")

Для `n = 12` правило `√n` дає приблизно `3,46`, а правило Стерджеса — приблизно `4,58`, тобто орієнтовно 3–4 та 5 інтервалів відповідно. Із трьох побудованих варіантів гістограма з 3 інтервалами найкраще показує загальну форму розподілу: 10 інтервалів надто подрібнюють малу вибірку, а стандартна гістограма може бути менш зручною для порівняння.

## Завдання 4. Міри положення й квантилі

In [ ]:
mean_value = precipitation.mean()
median_value = precipitation.median()
mode_values = precipitation.mode()
quartiles = precipitation.quantile([0.25, 0.75])

print("Середнє:", mean_value)
print("Медіана:", median_value)
print("Мода:", mode_values.to_list())
print("Q1:", quartiles.loc[0.25])
print("Q3:", quartiles.loc[0.75])

Для Києва середнє дорівнює `54,5 мм`, а медіана — `47,5 мм`. Різниця помітна: кілька вологих місяців із високими значеннями, зокрема 78 і 88 мм, підтягують середнє вгору. Це узгоджується з правосторонньою асиметрією розподілу опадів. Мода дорівнює `47 мм`, а квартилі приблизно `Q1 = 43,75 мм` і `Q3 = 62,5 мм`.

## Завдання 5. Зв'язок зі шкалою вимірювання

Кількість опадів належить до відносної шкали: є природний нуль, однакові одиниці вимірювання та змістовні відношення між значеннями. Тому для неї можна коректно застосовувати середнє, медіану й моду. Для змінної `місто`, яка є номінальною, числові коди були б лише умовними мітками, тому середнє та медіана не мали б змісту; коректною мірою положення залишається тільки мода.

## Письмові відповіді

1. Середнє арифметичне некоректне для номінальної змінної `місто`, навіть якщо міста закодувати числами, бо ці числа є довільними мітками. Зміна кодів змінила б середнє, хоча самі міста залишилися б тими самими.

2. Номер місяця має природний порядок і рівні календарні кроки, але не має природного нуля як відсутності місяця. Через це його можна трактувати як порядкову шкалу, а за іншого обґрунтування — як інтервальну календарну позицію. Класифікація залежить від мети аналізу.

3. Для кількості опадів майже всі 12 значень різні, тому стовпчикова діаграма за окремими значеннями була б перевантаженою. Гістограма об'єднує близькі значення в інтервали й показує форму розподілу.

4. Накопичена частота на межі третього інтервалу дорівнює 11. Це означає, що 11 із 12 місяців, або 91,7% року, мають опади не більші за 80 мм.

## Підсумок

Для варіанта Київ класифіковано змінні за шкалами вимірювання, побудовано варіаційний ряд, інтервальну частотну таблицю та гістограми з різною кількістю інтервалів. Обчислено середнє, медіану, моду й квартилі та пов'язано вибір мір положення з типом шкали.